In [1]:
from agent_pinner.pre_processing import *
import warnings

import pandas as pd
from pandas.core.common import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

In [2]:
import numpy as np
import gym
from gym import spaces
import random
from gym.utils import seeding



class conn(gym.Env):
       
  #Custom Environment that follows gym interface.

      # observation space
    #1 number pins [0,Num_Pin_unique_max]
    #2 distance to center pin [0,11]bucket
    #3 number of neighbors[0,Num_Pin_unique_max-1]
    #4 internal/external[0,1]
    #5 mean_neighbors [0,11]bucket
    #6 distance centroide [0,11]bucket
    #7 signal category [0,7]
    #8 color wire categorical[0,max_color_categories]
    #9 thickness wire continuous[0,51]bucket
    #10 multicore yes/no [0,1]
    #11 number of internal pins [0,Num_Pin_unique_max//2]
    #12 nosignal [0,1]
    #13 ismulticore [0,1]
    #14 category multicore [0,max_multicore_categories]

  # Because of google colab, we cannot implement the GUI ('human' render mode)
    metadata = {'render.modes': ['console']}
  # Define constants for clearer code


    def __init__(self, Num_Pin_unique_max, max_color_categories, max_categories_multicore, number_pins=100):

        super(conn, self).__init__()
        # numero de pins connector -1 due to PartNumber
        # space number of pins of this connector
        self.lae=14
        self.p=Num_Pin_unique_max
        self.c=max_color_categories
        self.m=max_categories_multicore
        self.action_space = spaces.Box(low=np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0]), \
                                      high=np.array([self.p+1,11,self.p,1,11,11,8,self.c+1,51,1,int(self.p//2),1,1, self.m+1]), \
                                       shape=(self.lae,))
        # observation space low/High bound number of diferent signals this connector
        # size, vector length number of pins
        self.observation_space = spaces.Box(low=np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0]), \
                                      high=np.array([self.p+1,11,self.p,1,11,11,8,self.c+1,51,1,int(self.p//2),1,1, self.m+1]), \
                                            shape=(self.lae,))
        # estate list of actions
        self.agent_pos=0
        self.bins10= np.linspace(1, 10, 11)
        self.bins100= np.linspace(0, 100, 101)
        self.binsp= np.linspace(0, self.p, self.p+1)# bin number of pins
        self.binsc= np.linspace(0, self.c, self.c+1)#  bin number of 
        self.binsm= np.linspace(0, self.m, self.m+1)
        self.number_pins=number_pins
        self.signal_in_conector=[]
        self.int_ext_in_conector=[]
        self.color_in_conector=[]
        self.number_pin_already_in_conector=[]
        self.distances_in_conector=[]
        self.no_signal_in_conector=[]
        self.reward=0
        return        

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def reset(self):
        self.state = np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0])#np.zeros(self.lae, dtype='int')
        self.signal_in_conector=[]
        self.int_ext_in_conector=[]
        self.color_in_conector=[]
        self.number_pin_already_in_conector=[]
        self.distances_in_conector=[]
        self.no_signal_in_conector=[]
        self.agent_pos=0
        self.reward=0
        return self.state

    def render(self, mode='console'):
        if mode != 'console':
            raise NotImplementedError()
          # agent is represented as a cross, rest as a dot

        return



    def get_reward_cat(self,signal_cat, int_ext,distance, no_signal_pin):
        # save the previous rewards
        reward=0
        if signal_cat in self.signal_in_conector and signal_cat != 7:
            reward = reward -1

        else:
            reward = reward + 4
        # higvoltage close in internal PIN or close to an empty cavity
        if (signal_cat in [1,6]) and int_ext == 1 :
            reward =reward +5
        elif (signal_cat in [1,6]) and no_signal_pin ==1:
            reward =reward +2
        else:
            reward =reward -5
            
        # if GROUND is present reward more when far from High Power categories
        if (signal_cat in [1,6]) and 4 in self.signal_in_conector:
            reward = reward - int((distance +10)//4)
        else:
            reward = reward + int((distance +10)//4)
            
        if (signal_cat in [1,6]) and 5 in self.signal_in_conector:
            reward = reward - int((distance +10)//4)
        else:
            reward = reward + int((distance +10)//4)
        # if High power     is present reward more when far from ground categories
        if (signal_cat in [4,5]) and 1 in self.signal_in_conector:
            reward = reward - int((distance +10)//4)
        else:
            reward = reward + int((distance +10)//4)
            
        if (signal_cat in [4,5]) and 6 in self.signal_in_conector:
            reward = reward - int((distance +10)//4)
        else:
            reward = reward + int((distance +10)//4)
        #reward more high power far from other signals
        if signal_cat ==6 and len(self.distances_in_conector)>1:
            for ele in self.distances_in_conector:
                diff=abs(distance-ele)
                reward = reward + int((diff +10)//4)

        

        return reward

    def get_reward_color(self, color_cat):
        # save the previous rewards
        reward=0
        if color_cat in self.color_in_conector:
            reward = reward -4

        else:
            reward = reward + 4
        return reward

    def get_reward_int_ext(self, number_internal):
        # save the previous rewards
        reward=0
        if number_internal>  len(self.number_pin_already_in_conector):
            reward = reward -4

        else:
            reward = reward + 4
        return reward
    
    def get_reward_multicore_out(self, distance, multicore):
        # save the previous rewards
        reward=0
        if multicore ==1:
            reward = reward + int((distance +10)//2) 

        else:
            reward = reward - int((distance +10)//2)
        return reward
    
    def get_bin_s(self,n, bins):
        _bin=0
        for i in range(0,len(bins)):
            if bins[i]>= n:
                _bin=i-1
                break
        return _bin
    
    def get_zero_one(self, n):
        if n < 0.1:
            return 0
        else:
            return 1
    def get_reward_empty_cavities(self, number_of_empty_cavities):
        reward =0
        # if high power in connector reward more empty cavities 
        if 6 in self.signal_in_conector or 1 in self.signal_in_conector:
            reward = int((10 + number_of_empty_cavities )//10)
        elif number_of_empty_cavities <2 :
            reward = reward - 2
                
        return reward
        
    def calculate_reward(self,action):
        reward=0
        distance=self.get_bin_s(action[1],self.bins10) # distance to the center
        int_ext= self.get_zero_one(action[3]) # internal External Flag
        signal_cat=int(action[6]) # signal Category
        color_cat=int(action[7]) # color Category
        multicore=self.get_zero_one(action[9]) # multicore
        number_internal=int(action[10]) # number of internal pins
        no_signal_pin=self.get_zero_one(action[11]) # is an empty cavity
        number_of_empty_cavities=self.no_signal_in_conector.count(1) # number of empty cavities

        #print(distance)
        reward = reward + self.get_reward_cat( signal_cat, int_ext, distance,no_signal_pin)
        reward = reward + self.get_reward_color(color_cat)
        reward = reward + self.get_reward_int_ext(number_internal)
        reward = reward + self.get_reward_multicore_out(distance, multicore )
        reward = reward + self.get_reward_empty_cavities(number_of_empty_cavities )
        # append what you see to the observation space
        self.signal_in_conector.append(signal_cat)
        self.int_ext_in_conector.append(int_ext)
        self.color_in_conector.append(color_cat)
        self.distances_in_conector.append(distance)
        self.no_signal_in_conector.append(no_signal_pin)
        if int_ext==1:
            self.number_pin_already_in_conector.append(number_internal)
        
        return reward




    def step(self, action):
        
        #real=self.get_real(action) #  real values
        #calculate Reward
        self.reward = self.reward+self.calculate_reward(action)
        #print(self.reward)


        # Account for the boundaries of the grid
        self.agent_pos =  self.agent_pos + 1
        self.terminal = bool(self.agent_pos > self.p) 

        # Optionally we can pass additional info, we are not using that for now
        info = {}

        return self.state, self.reward, self.terminal, info


In [3]:
test=np.linspace(1, 12, 11)
test

array([ 1. ,  2.1,  3.2,  4.3,  5.4,  6.5,  7.6,  8.7,  9.8, 10.9, 12. ])

In [4]:
data1 = pd.read_csv("agent_pinner/data/symbol_pins.csv")
symbols=get_neighbors_partnumbers(data1)

data = pd.read_csv("agent_pinner/data/inline_pins.csv")
data, d, _ = preprocess_data(data, symbols)

connectors_dicc=create_connector_dicc(data)
#connector="TO_111_LH_F/143_3P"

In [8]:
len(data['Connector Name'].unique())

60

In [15]:
data.columns

Index(['Connector Name', 'Pin Name', 'Pin Name nunique', 'Pin Name count',
       'Connector PartNumber', 'Pin PreferredSignal', 'Signal Name',
       'Wire WireColor', 'Wire WireCSA', 'MulticoreInnerToOutter1',
       'MulticoreInnerToOutter2', 'MulticoreInnerToOutter4',
       'MulticoreInnerToOutter3', 'MulticoreInnerToOutter5',
       'MulticoreInnerToOutter6', 'MulticoreInnerToOutter7',
       'MulticoreInnerToOutter8', 'MulticoreInnerToOutter9', 'check_signal',
       'Num_Pin_unique', 'Num_Pins', 'more_signal_in_pin', 'signal_cat',
       'signal_multicore', 'Cat_Signal Name', 'dicc_signals',
       'Cat_PreferredSignal', 'Cat_Wire_WireColor', 'Cat_Wire_WireColor_max',
       'Wire_WireCSA_min', 'Wire_WireCSA_max', 'Num_Pin_unique_max',
       'no_signal_pin', 'multicore_same', 'is_multicore',
       'max_categories_multicore', 'Symbol Name', 'Pin CenterY', 'Pin CenterX',
       'Pin Width', 'Pin Height', 'distance_to_center', 'neighbors',
       'neighbors_length', 'internal_pi

In [10]:
connector='TO_169_F/161_RH_13P'

In [28]:
d_models=[]
model_connectors={}

for connector in connectors_dicc.keys():
    num_pins=connectors_dicc[connector]['1']['Num_Pins']
    
    if not(num_pins in d_models):
        d_models.append(num_pins)
        
        model_connectors[num_pins]=[]
    if not(connector in model_connectors[num_pins]):
         model_connectors[num_pins].append(connector)
        
d_models


[17,
 28,
 3,
 53,
 49,
 50,
 41,
 35,
 42,
 10,
 8,
 13,
 2,
 18,
 55,
 31,
 52,
 25,
 24,
 19,
 38,
 47,
 21,
 5]

In [40]:
model_connectors[28]#'TO_111_LH_F/121'

['TO_111_LH_F/121', 'TO_121_F/111_LH', 'TO_141_LH_F/152', 'TO_152_F/141_LH']

In [151]:
def get_bin_10(n,bins10):
    _bin=10
    for i in range(0,len(bins10)):
        if bins10[i]> n:
            _bin=1
            break
    return _bin

In [152]:
Num_Pin_unique_max=connectors_dicc[connector]['1']['Num_Pin_unique_max']
Num_Pin_unique_max

55

In [153]:
max_color_categories=Cat_Wire_WireColor=connectors_dicc[connector]['1']['Cat_Wire_WireColor_max']
max_color_categories

30

In [154]:
max_categories_multicore=Cat_Wire_WireColor=connectors_dicc[connector]['1']['max_categories_multicore']
max_categories_multicore

8

In [155]:
Num_Pins= connectors_dicc[connector]['1']['Num_Pins']
Num_Pins

13

In [156]:
def get_pin_signal(sig_conn,i):
    bins_distance= np.linspace(sig_conn['1']['min_distance'], sig_conn['1']['max_distance'], 11)
    bins_thickness= np.linspace(sig_conn['1']['Wire_WireCSA_min'], sig_conn['1']['Wire_WireCSA_max'], 51)
    Num_Pins=sig_conn[str(i)]['Num_Pins']
    distance_to_center=get_bin_10(sig_conn[str(i)]['distance_to_center'], bins_distance)
    neighbors_length=sig_conn[str(i)]['neighbors_length']
    internal_pin = 1 if sig_conn[str(i)]['internal_pin'] =='True' else 0
    mean_neighbors=get_bin_10(sig_conn[str(i)]['mean_neighbors'], bins_distance)
    centroide_distance=get_bin_10(sig_conn[str(i)]['centroide_distance'], bins_distance)
    signal_group=sig_conn[str(i)]['signal_group']
    Cat_Wire_WireColor=sig_conn[str(i)]['Cat_Wire_WireColor']
    WireCSA=get_bin_10(sig_conn[str(i)]['WireCSA'], bins_thickness)
    is_multicore = 0 if sig_conn[str(i)]['is_multicore'] =='na' else 1
    number_internal_pin= sig_conn[str(i)]['number_internal_pin']
    nosignal = sig_conn[str(i)]['no_signal_pin']
    ismulticore= sig_conn[str(i)]['is_multicore']
    cat_multicore=sig_conn[str(i)]['multicore_same']
    return list([Num_Pins,distance_to_center,neighbors_length,internal_pin,mean_neighbors,
                 centroide_distance,signal_group,Cat_Wire_WireColor,WireCSA,is_multicore,
                 number_internal_pin,nosignal,ismulticore, cat_multicore])

In [157]:
get_pin_signal(connectors_dicc[connector],1)

[13, 10, 3, 0, 1, 1, 6, 26, 1, 1, 2.0, 0, 0, 1]

In [158]:
#action space == number of PIN,s

    # observation space
   #1 number pins [0,Num_Pin_unique_max]
    #2 distance to center pin [0,11]bucket
    #3 number of neighbors[0,Num_Pin_unique_max-1]
    #4 internal/external[0,1]
    #5 mean_neighbors [0,11]bucket
    #6 distance centroide [0,11]bucket
    #7 signal category [0,7]
    #8 color wire categorical[0,max_color_categories]
    #9 thickness wire continuous[0,51]bucket
    #10 multicore yes/no [0,1]
    #11 number of internal pins [0,Num_Pin_unique_max//2]
    #12 nosignal [0,1]
    #13 ismulticore [0,1]
    #14 category multicore [0,max_multicore_categories]



In [159]:
from stable_baselines3.common.env_checker import check_env

In [206]:
env = conn(Num_Pin_unique_max, max_color_categories,max_categories_multicore, Num_Pins) 
obs = env.reset()
env.render()

In [207]:
check_env(env, warn=True)

C:\Users\User\AppData\Roaming\Python\Python38\site-packages\stable_baselines3\common\env_checker.py:231: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(


In [162]:
print(env.observation_space)
print(env.action_space)
print(env.action_space.sample())

Box(0.0, 56.0, (14,), float32)
Box(0.0, 56.0, (14,), float32)
[27.033718    7.2732325  53.257896    0.34909415  0.97068226  5.181375
  3.2085385  15.310357   28.473034    0.06615737 25.46332     0.46346766
  0.5584964   2.4409416 ]


In [163]:
import sys

In [164]:
def gen_step():
    step=[]
    step.append(random.randint(0, Num_Pin_unique_max+1 ))
    step.append(random.randint(0,11))
    step.append(random.randint(0, Num_Pin_unique_max ))
    step.append(random.randint(0, 1 ))
    step.append(random.randint(0,11))
    step.append(random.randint(0,11))
    step.append(random.randint(0, 8 ))
    step.append(random.randint(0, max_color_categories ))
    step.append(random.randint(0,11))
    step.append(random.randint(0,1))
    step.append(random.randint(0,1))
    step.append(random.randint(0,int(Num_Pin_unique_max//2)))
    step.append(random.randint(0,1))
    step.append(random.randint(0,1))
    step.append(random.randint(0, max_categories_multicore ))
    return step
    
#[self.p+1,11,self.p,1,11,11,8,self.c+1,51,1,int(self.p//2)]    

In [165]:
obs = env.reset()
env.render()
n_steps = 1300
#GO_LEFT=random.randint(0, max_categories )
for step in range(n_steps):
    #print("Step {}".format(step + 1))
    GO_LEFT=gen_step()
    obs, reward, done, info = env.step(GO_LEFT)
    #print('obs=', obs, 'reward=', reward, 'done=', done)
    env.render()
    if done:
        print("Goal reached!", "reward=", reward)
        obs = env.reset()

8
16
27
53
89
121
141
147
175
175
195
198
194
216
266
273
285
293
366
441
465
542
533
535
545
633
655
658
742
752
766
795
812
827
936
1055
1070
1097
1094
1104
1233
1249
1260
1278
1301
1476
1480
1495
1636
1656
1646
1666
1671
1863
1888
1888
Goal reached! reward= 1888
24
40
40
61
64
79
98
113
157
178
211
221
244
250
246
247
255
324
385
390
396
395
395
405
434
445
462
476
492
504
517
529
541
550
567
572
575
580
595
601
614
628
651
662
673
843
851
863
874
872
888
1046
1049
1058
1050
1057
Goal reached! reward= 1057
22
31
38
49
59
83
96
91
102
105
114
135
186
202
206
213
215
248
257
248
257
256
274
275
279
296
325
336
345
361
375
398
405
435
438
454
467
624
626
617
613
615
759
749
762
768
785
803
819
825
972
995
1001
1019
1022
1013
Goal reached! reward= 1013
22
43
58
89
106
115
133
148
179
197
200
209
217
206
228
242
258
264
274
284
298
382
389
411
412
501
506
504
590
590
602
618
624
638
661
684
701
704
729
740
758
748
739
749
753
761
773
802
820
833
838
851
871
874
893
918
Goal reached! rewa

In [166]:
from stable_baselines3.common.vec_env import DummyVecEnv
env_train = DummyVecEnv([lambda: conn(Num_Pin_unique_max, max_color_categories, max_categories_multicore, Num_Pins)])

In [167]:
from stable_baselines3 import DQN, TD3, PPO, A2C, SAC, DDPG
model = SAC('MlpPolicy', env_train, verbose=2, create_eval_env=True)

Using cpu device


In [168]:
import time
time1=time.time()
model.learn(10005)
time2=time.time()

print(time2-time1)

9
14
21
32
50
63
82
86
99
111
162
190
191
200
201
228
222
245
246
275
294
314
325
335
357
444
459
489
516
539
553
546
659
675
689
701
832
854
876
887
909
910
926
957
974
978
983
1012
1019
1205
1222
1229
1255
1266
1460
1483
9
12
24
42
49
61
67
77
115
128
147
197
229
237
263
267
264
252
275
284
296
311
324
333
335
350
366
383
403
424
445
546
657
672
690
699
708
718
747
767
770
781
805
835
859
876
894
916
922
937
944
967
991
999
1003
1010
2
24
42
46
71
107
105
110
137
149
160
164
164
159
154
181
196
196
208
228
247
269
281
283
365
382
398
410
439
447
448
453
469
479
482
484
494
495
518
534
545
544
574
601
618
760
767
786
792
795
818
835
845
862
865
878
14
33
49
68
86
99
108
102
108
111
132
137
156
178
185
197
231
250
259
278
282
304
326
411
426
518
526
548
553
583
601
601
616
741
768
778
794
784
801
807
818
839
842
859
867
878
907
939
945
957
962
999
1016
1186
1200
1206
---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps          

0
-8
-8
-16
-24
-22
-20
-28
-26
-34
-30
-36
-32
-28
-24
-30
-36
8
3
-2
-11
-6
30
25
21
17
22
17
74
143
157
163
228
239
243
257
340
340
350
365
380
380
385
400
415
415
430
445
445
450
460
488
499
515
526
527
10
12
4
3
-3
-4
0
10
15
5
10
15
5
0
-8
-17
-11
-24
-18
-17
-16
-25
-30
-31
-36
-35
-31
-21
-11
11
7
7
7
7
12
16
21
21
32
37
59
69
74
90
112
114
124
135
139
140
152
164
175
187
194
218
10
5
3
-8
-9
-11
-22
-23
-24
-26
-28
-30
-40
-51
-61
-63
-64
-66
-77
-78
-88
-99
-110
-121
-122
-111
-113
-115
-126
-137
-138
-148
-138
-148
-158
-168
-161
-164
-158
-151
-154
-148
-150
-143
-137
-139
-127
-121
-114
-108
-102
-96
-89
-82
-76
-64
----------------------------------
| time/              |           |
|    episodes        | 28        |
|    fps             | 19        |
|    time_elapsed    | 81        |
|    total timesteps | 1568      |
| train/             |           |
|    actor_loss      | -3.67e+03 |
|    critic_loss     | 2.5e+05   |
|    ent_coef        | 1.48      |
|    ent_coef

15
28
33
38
43
40
37
34
39
44
49
54
59
64
69
66
71
68
73
86
91
96
93
98
103
108
113
118
123
128
133
138
143
148
153
150
155
160
165
170
175
180
177
182
187
192
197
202
207
212
209
214
219
224
229
----------------------------------
| time/              |           |
|    episodes        | 52        |
|    fps             | 18        |
|    time_elapsed    | 154       |
|    total timesteps | 2912      |
| train/             |           |
|    actor_loss      | -6.98e+03 |
|    critic_loss     | 5.86e+05  |
|    ent_coef        | 2.76      |
|    ent_coef_loss   | -122      |
|    learning_rate   | 0.0003    |
|    n_updates       | 2811      |
----------------------------------
18
23
34
31
36
41
46
51
56
61
74
79
84
89
94
99
104
109
114
119
124
129
134
139
144
149
154
159
164
169
174
171
176
181
186
191
204
217
222
227
232
237
242
247
252
257
262
267
272
269
274
279
284
289
294
299
18
23
28
33
38
43
48
53
58
63
68
73
83
88
93
98
111
116
129
134
139
144
149
154
159
164
169
174
187
192
19

176
182
188
194
200
211
222
228
226
237
235
241
238
249
255
261
259
266
272
262
260
266
----------------------------------
| time/              |           |
|    episodes        | 76        |
|    fps             | 18        |
|    time_elapsed    | 225       |
|    total timesteps | 4256      |
| train/             |           |
|    actor_loss      | -7.96e+03 |
|    critic_loss     | 1.24e+06  |
|    ent_coef        | 4.32      |
|    ent_coef_loss   | -139      |
|    learning_rate   | 0.0003    |
|    n_updates       | 4155      |
----------------------------------
17
21
19
30
27
33
23
21
27
25
39
37
48
54
60
74
85
99
105
119
117
123
129
127
133
139
137
143
149
147
153
159
165
176
174
195
205
216
226
236
238
248
259
266
285
296
299
315
326
338
354
365
369
381
393
405
17
29
35
46
52
58
56
70
76
90
96
110
116
122
128
139
137
143
149
155
161
172
178
184
190
196
202
208
214
220
226
232
238
244
250
261
275
281
307
305
311
317
315
321
332
343
341
339
350
356
370
381
392
398
396
402
17


17
26
35
39
35
41
60
66
77
88
94
100
103
109
107
110
113
119
122
120
126
129
135
146
152
150
148
143
141
148
159
165
168
174
188
279
288
297
315
324
333
334
343
359
377
392
394
400
415
417
432
434
441
444
447
463
9
18
27
28
29
40
51
46
65
63
112
116
123
122
129
136
135
139
143
143
159
167
180
193
209
222
227
227
232
232
241
242
251
257
266
275
292
306
307
404
418
427
433
443
453
464
474
484
486
488
498
501
508
526
534
546
9
13
14
15
16
19
25
31
37
40
38
49
54
67
74
127
134
142
155
168
176
176
181
201
222
230
238
246
259
267
280
296
305
306
314
328
329
338
360
361
362
379
394
403
412
434
444
459
469
471
482
497
503
505
507
522
9
18
30
41
44
50
56
70
76
82
80
86
85
91
97
103
109
209
217
217
222
230
236
245
251
260
266
280
350
373
374
383
383
389
403
404
405
414
416
431
433
443
450
473
480
482
497
507
522
537
539
563
574
590
601
604
----------------------------------
| time/              |           |
|    episodes        | 104       |
|    fps             | 18        |
|    time_elapsed 

87
98
109
115
114
112
110
10
14
20
26
33
30
42
48
37
27
34
24
23
35
33
37
34
48
52
47
50
40
44
58
48
52
63
62
68
75
70
59
65
55
52
51
41
47
53
43
67
71
70
61
59
57
64
62
61
55
61
72
70
68
73
68
9
13
20
27
33
45
56
54
43
49
56
55
53
52
50
49
47
49
64
62
60
63
69
67
74
84
82
80
84
82
81
93
99
105
103
101
111
109
108
115
114
112
110
108
98
97
96
86
84
82
101
99
97
102
101
104
1
-4
2
8
7
7
18
26
25
24
23
31
39
59
51
52
61
61
62
54
69
77
82
82
91
107
109
111
121
135
137
138
140
150
160
169
172
165
172
166
169
180
190
191
199
210
212
214
217
228
245
257
260
272
267
262
----------------------------------
| time/              |           |
|    episodes        | 128       |
|    fps             | 18        |
|    time_elapsed    | 382       |
|    total timesteps | 7168      |
| train/             |           |
|    actor_loss      | -4.01e+03 |
|    critic_loss     | 2.61e+05  |
|    ent_coef        | 8.88      |
|    ent_coef_loss   | -90.6     |
|    learning_rate   | 0.0003    |
|    n_upd

1087
1105
1123
1153
1175
1193
1363
17
41
66
79
91
112
130
148
161
174
193
221
246
260
311
324
337
350
364
389
475
489
599
613
628
730
749
768
783
798
814
835
854
889
917
944
965
980
996
1012
1035
1063
1091
1114
1130
1158
1174
1191
1213
1235
1252
1282
1305
1335
1353
1370
22
32
52
69
89
113
137
153
178
190
202
214
238
250
268
280
304
316
328
340
352
372
384
408
420
443
475
499
517
529
541
553
565
569
581
593
704
720
736
752
781
798
815
832
849
866
883
905
933
955
973
1003
1032
1062
1080
1098
18
39
63
95
115
139
151
175
187
239
258
284
298
312
326
359
385
399
413
439
454
481
496
522
537
552
573
608
623
638
666
690
711
727
743
771
786
802
817
833
850
866
889
906
935
957
974
997
1026
1043
1066
1089
1106
1124
1154
1172
----------------------------------
| time/              |           |
|    episodes        | 176       |
|    fps             | 18        |
|    time_elapsed    | 526       |
|    total timesteps | 9856      |
| train/             |           |
|    actor_loss      | -6.91e+03

In [169]:
from stable_baselines3.common.evaluation import evaluate_policy

In [170]:
env = conn(Num_Pin_unique_max, max_color_categories,max_categories_multicore, Num_Pins) 
obs = env.reset()
env.render()
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f'Mean reward: {mean_reward} +/- {std_reward:.2f}')

C:\Users\User\AppData\Roaming\Python\Python38\site-packages\stable_baselines3\common\evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


18
28
40
52
64
76
88
100
112
124
136
148
160
172
184
196
208
220
232
244
256
268
280
292
304
316
328
340
352
364
376
388
400
412
424
436
448
460
472
484
496
508
520
532
544
556
568
580
592
604
616
628
640
652
664
676
18
28
40
52
64
76
88
100
112
124
136
148
160
172
184
196
208
220
232
244
256
268
280
292
304
316
328
340
352
364
376
388
400
412
424
436
448
460
472
484
496
508
520
532
544
556
568
580
592
604
616
628
640
652
664
676
18
28
40
52
64
76
88
100
112
124
136
148
160
172
184
196
208
220
232
244
256
268
280
292
304
316
328
340
352
364
376
388
400
412
424
436
448
460
472
484
496
508
520
532
544
556
568
580
592
604
616
628
640
652
664
676
18
28
40
52
64
76
88
100
112
124
136
148
160
172
184
196
208
220
232
244
256
268
280
292
304
316
328
340
352
364
376
388
400
412
424
436
448
460
472
484
496
508
520
532
544
556
568
580
592
604
616
628
640
652
664
676
18
28
40
52
64
76
88
100
112
124
136
148
160
172
184
196
208
220
232
244
256
268
280
292
304
316
328
340
352
364
376
388
400
412
424

In [171]:
connector='TO_121_TNGA_F/219_13P'

In [172]:
env = conn(Num_Pin_unique_max, max_color_categories,max_categories_multicore, Num_Pins) 

In [173]:
obs = env.reset()
for key in connectors_dicc[connector].keys():
    if key != 'PartNumber':
        obs = np.array(get_pin_signal(connectors_dicc[connector],key))
        action, _state = model.predict(obs, deterministic=True)
        
        obs, reward, done, info = env.step(action.astype(int))
        print(reward)
        env.render()
        if done:
           obs = env.reset()
print(reward)

18
18
22
22
27
27
28
28
35
35
42
42
43
43
50
50
51
51
58
58
65
65
72
72
79
79
79


In [174]:
#TD3 'TO_169_F/161_RH_13P' = 123 , 'TO_121_TNGA_F/219_13P' =169  'TO_169_F/161_RH_13P' = 134 'TO_111_LH_F/143_3P' 227

In [175]:
# for key in connectors_dicc[connector].keys():
#     if key != 'PartNumber':
#         pred= np.array(get_pin_signal(connectors_dicc[connector],key))
        
#         prediction=model.predict(pred,  deterministic= False)
        

In [176]:
Num_Pins

13

In [177]:
connectors_dicc[connector]

{'1': {'PreferredSignal': 'na',
  'Cat_PreferredSignal': 27,
  'Signal': 'SLD_EFI_ECU-TNGA_KNOCK_SSR_101',
  'Cat_Signal': 14,
  'dicc_signals': "{0: 'CLEARANCE_SNR_ECU_0885', 1: 'CLEARANCE_SNR_ECU_0890', 2: 'EQ_FUEL_PRESS_SSR_HI-TNGA_E2', 3: 'EQ_FUEL_PRESS_SSR_HI-TNGA_PR', 4: 'EQ_FUEL_PRESS_SSR_HI-TNGA_VC', 5: 'EQ_NE_SSR-TNGA_NE-', 6: 'EQ_NE_SSR-TNGA_NEP', 7: 'EQ_NE_SSR-TNGA_VCNE', 8: 'EQ_PDB_L_1', 9: 'EQ_PDB_L_2', 10: 'EQ_PDB_L_5', 11: 'GND_BK_UP_LP_LH_E', 12: 'HV_ECU-HV_0024', 13: 'HV_ECU-HV_0025', 14: 'SLD_EFI_ECU-TNGA_KNOCK_SSR_101', 15: 'SLD_EFI_ECU-TNGA_KNOCK_SSR_102', 16: 'SLD_MG_ECU_197', 17: 'SLD_MG_ECU_198', 18: 'SLD_MG_ECU_199', 19: 'SLD_MG_ECU_200', 20: 'SLD_MG_ECU_201', 21: 'SLD_MG_ECU_202', 22: 'SLD_MG_ECU_203', 23: 'TW_CLEARANCE_SNR_ECU_055', 24: 'TW_CLEARANCE_SNR_ECU_056', 25: 'TW_CLEARANCE_SNR_ECU_061', 26: 'TW_CLEARANCE_SNR_ECU_062', 27: 'na'}",
  'check_signal': 0,
  'Num_Pins': 13,
  'Num_Pin_unique': 13,
  'more_signal_in_pin': False,
  'WireColor': 'W',
  'WireCS

In [179]:
dicc_13={0: 'CLEARANCE_SNR_ECU_0885', 1: 'CLEARANCE_SNR_ECU_0890', 2: 'EQ_FUEL_PRESS_SSR_HI-TNGA_E2', 3: 'EQ_FUEL_PRESS_SSR_HI-TNGA_PR', 4: 'EQ_FUEL_PRESS_SSR_HI-TNGA_VC', 5: 'EQ_NE_SSR-TNGA_NE-', 6: 'EQ_NE_SSR-TNGA_NEP', 7: 'EQ_NE_SSR-TNGA_VCNE', 8: 'EQ_PDB_L_1', 9: 'EQ_PDB_L_2', 10: 'EQ_PDB_L_5', 11: 'GND_BK_UP_LP_LH_E', 12: 'HV_ECU-HV_0024', 13: 'HV_ECU-HV_0025', 14: 'SLD_EFI_ECU-TNGA_KNOCK_SSR_101', 15: 'SLD_EFI_ECU-TNGA_KNOCK_SSR_102', 16: 'SLD_MG_ECU_197', 17: 'SLD_MG_ECU_198', 18: 'SLD_MG_ECU_199', 19: 'SLD_MG_ECU_200', 20: 'SLD_MG_ECU_201', 21: 'SLD_MG_ECU_202', 22: 'SLD_MG_ECU_203', 23: 'TW_CLEARANCE_SNR_ECU_055', 24: 'TW_CLEARANCE_SNR_ECU_056', 25: 'TW_CLEARANCE_SNR_ECU_061', 26: 'TW_CLEARANCE_SNR_ECU_062', 27: 'na'}

In [180]:
num_signals=len(dicc_13.keys())
Num_Pins, num_signals

(13, 28)

In [181]:
import random
import pandas as pd

In [182]:
cols=[str(i) for i in range(1,Num_Pins+1 )]

In [183]:
data_test= pd.DataFrame(data=[], columns=cols)
data_test.head()

,1,2,3,4,5,6,7,8,9,10,11,12,13


In [184]:
for x in range(0,500):
    variant=[]
    data={}
    for i in range(0, Num_Pins):
        variant.append(random.randint(0, num_signals-1))
        data[str(i+1)]=random.randint(0, num_signals-1)
    data_test= data_test.append(data, ignore_index=True)
    print(variant)

[24, 13, 4, 0, 17, 19, 11, 27, 20, 27, 25, 24, 4]
[11, 20, 6, 12, 25, 13, 21, 24, 3, 10, 13, 6, 10]
[8, 12, 24, 0, 11, 0, 11, 12, 4, 7, 3, 8, 12]
[1, 26, 10, 14, 0, 0, 18, 7, 13, 13, 15, 21, 18]
[15, 20, 22, 0, 26, 6, 6, 5, 2, 1, 20, 12, 12]
[11, 1, 7, 7, 21, 20, 17, 11, 11, 4, 23, 12, 22]
[25, 4, 16, 11, 6, 19, 12, 4, 24, 21, 7, 10, 18]
[23, 11, 12, 22, 9, 14, 22, 10, 9, 20, 11, 8, 0]
[12, 15, 22, 6, 8, 16, 23, 19, 8, 26, 27, 22, 12]
[12, 23, 12, 16, 13, 0, 19, 16, 11, 21, 4, 0, 27]
[20, 19, 8, 11, 22, 16, 16, 18, 1, 7, 22, 12, 6]
[16, 14, 12, 9, 23, 4, 10, 14, 19, 20, 17, 23, 21]
[6, 2, 6, 24, 17, 13, 4, 12, 11, 6, 5, 6, 8]
[7, 5, 22, 14, 21, 25, 12, 2, 0, 11, 20, 22, 12]
[7, 3, 24, 7, 1, 1, 27, 18, 6, 7, 17, 5, 23]
[25, 8, 15, 15, 24, 27, 4, 17, 27, 11, 9, 21, 27]
[13, 24, 11, 26, 9, 19, 12, 6, 26, 17, 4, 17, 20]
[24, 24, 20, 13, 12, 3, 13, 25, 15, 6, 8, 16, 5]
[10, 19, 7, 23, 13, 13, 13, 20, 1, 19, 11, 24, 1]
[26, 27, 0, 24, 26, 25, 0, 17, 1, 15, 24, 25, 8]
[7, 23, 5, 12, 17, 15, 2

[2, 21, 26, 23, 22, 26, 15, 5, 26, 7, 27, 16, 4]
[9, 23, 13, 17, 19, 17, 10, 6, 15, 9, 11, 7, 20]
[0, 14, 16, 26, 26, 14, 22, 21, 15, 21, 9, 7, 2]
[3, 17, 19, 3, 22, 26, 10, 1, 0, 0, 21, 11, 3]
[24, 26, 24, 2, 26, 20, 1, 24, 0, 17, 4, 15, 11]
[14, 3, 6, 20, 0, 27, 23, 27, 14, 6, 14, 8, 10]
[14, 11, 25, 22, 15, 1, 4, 24, 19, 22, 5, 14, 4]
[11, 2, 22, 1, 16, 11, 13, 6, 13, 12, 10, 6, 17]
[15, 9, 1, 15, 27, 3, 26, 21, 22, 22, 20, 9, 1]
[25, 8, 9, 7, 10, 15, 26, 11, 15, 10, 26, 10, 4]
[9, 25, 20, 20, 14, 18, 18, 12, 18, 11, 20, 2, 19]
[0, 15, 3, 11, 18, 24, 24, 5, 18, 9, 19, 7, 12]
[9, 7, 13, 15, 22, 0, 3, 16, 27, 17, 1, 27, 20]
[14, 9, 3, 24, 22, 12, 14, 27, 0, 19, 22, 21, 18]
[15, 9, 10, 15, 7, 9, 18, 8, 24, 12, 18, 4, 23]
[16, 19, 16, 26, 27, 21, 7, 9, 24, 3, 23, 22, 5]
[13, 26, 16, 8, 22, 20, 23, 14, 19, 22, 19, 4, 2]
[2, 8, 1, 19, 23, 27, 18, 21, 5, 17, 15, 3, 12]
[21, 17, 10, 6, 21, 14, 27, 21, 15, 22, 27, 4, 13]
[24, 23, 4, 11, 12, 21, 23, 4, 11, 10, 2, 16, 25]
[26, 26, 21, 27, 23, 

[2, 25, 18, 14, 20, 17, 12, 14, 23, 25, 2, 23, 13]
[16, 4, 3, 20, 5, 15, 0, 4, 21, 9, 27, 22, 15]
[7, 3, 10, 5, 11, 10, 4, 12, 1, 11, 19, 1, 0]
[19, 27, 5, 24, 26, 10, 2, 3, 19, 15, 5, 5, 19]
[18, 15, 14, 22, 17, 5, 6, 13, 21, 4, 18, 6, 16]
[12, 24, 3, 21, 12, 10, 20, 23, 20, 27, 11, 27, 23]
[17, 9, 25, 19, 23, 21, 14, 27, 15, 17, 25, 23, 25]
[25, 8, 26, 6, 20, 27, 10, 18, 15, 0, 25, 17, 9]
[9, 27, 21, 22, 16, 21, 14, 6, 19, 10, 7, 15, 20]
[13, 15, 6, 20, 2, 2, 20, 1, 12, 13, 17, 7, 26]
[12, 6, 17, 6, 6, 10, 6, 2, 19, 25, 12, 14, 4]
[18, 11, 14, 20, 13, 4, 15, 3, 15, 20, 12, 11, 14]
[17, 24, 3, 7, 10, 27, 13, 25, 14, 16, 15, 6, 21]
[10, 2, 15, 23, 16, 16, 27, 10, 26, 4, 4, 4, 19]
[20, 19, 10, 16, 6, 16, 22, 4, 0, 10, 23, 18, 5]
[27, 10, 20, 14, 24, 19, 12, 24, 2, 15, 27, 8, 11]
[13, 21, 3, 7, 1, 6, 23, 7, 27, 6, 26, 27, 5]
[20, 24, 12, 7, 11, 20, 20, 4, 1, 11, 14, 13, 18]
[1, 7, 27, 15, 17, 8, 24, 16, 4, 15, 14, 27, 11]
[22, 20, 3, 7, 8, 23, 26, 11, 16, 8, 20, 4, 22]
[13, 6, 27, 13, 10

In [185]:
data_test.head()

,1,2,3,4,5,6,7,8,9,10,11,12,13
0,5,14,16,5,26,14,24,24,15,27,15,13,23
1,17,7,25,10,14,9,6,26,5,19,22,26,24
2,12,8,20,7,7,0,10,22,8,12,11,9,0
3,7,19,19,6,13,7,16,26,11,8,9,15,12
4,16,21,2,20,10,8,5,11,27,20,18,0,3


In [186]:
def category_signal_test(row,col):
    """

    :param row:
    :return:
    """
    if row["cat_"+str(col)][:5]=='BATT_':
        return 1
    elif row["cat_"+str(col)][:4]=='CAN_':
        return 2
    elif row["cat_"+str(col)][:6]=='CANFD_':
        return 3
    elif row["cat_"+str(col)][:4]=='GND_':
        return 4
    elif row["cat_"+str(col)][:9]=='GNDR_GND_':
        return 5
    elif row["cat_"+str(col)][:3]=='HV_':
        return 6
    else:
        return 7


def return_cat(row, col ,dicc_cat):
    cat=row[col]
    return dicc_cat[cat]

In [187]:
for col in data_test.columns:
    data_test["cat_"+str(col)]= data_test.apply(lambda row: return_cat(row, col ,dicc_13), axis=1)
    data_test[col]= data_test.apply(lambda row: category_signal_test(row, col ), axis=1)

In [188]:
data_test.tail()

,1,2,3,4,5,6,7,8,9,10,...,cat_4,cat_5,cat_6,cat_7,cat_8,cat_9,cat_10,cat_11,cat_12,cat_13
495,7,7,6,7,7,7,7,7,7,7,...,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,SLD_MG_ECU_201,TW_CLEARANCE_SNR_ECU_056,EQ_FUEL_PRESS_SSR_HI-TNGA_VC,SLD_EFI_ECU-TNGA_KNOCK_SSR_101,EQ_NE_SSR-TNGA_NE-,SLD_MG_ECU_198,EQ_PDB_L_5,HV_ECU-HV_0025,SLD_MG_ECU_203
496,7,7,7,7,7,7,4,4,7,7,...,na,SLD_MG_ECU_199,SLD_MG_ECU_199,GND_BK_UP_LP_LH_E,GND_BK_UP_LP_LH_E,SLD_MG_ECU_199,SLD_MG_ECU_201,CLEARANCE_SNR_ECU_0890,CLEARANCE_SNR_ECU_0890,EQ_NE_SSR-TNGA_NE-
497,7,7,7,7,7,7,7,7,7,7,...,SLD_MG_ECU_200,TW_CLEARANCE_SNR_ECU_062,EQ_NE_SSR-TNGA_VCNE,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,TW_CLEARANCE_SNR_ECU_055,SLD_MG_ECU_197,EQ_FUEL_PRESS_SSR_HI-TNGA_E2,SLD_MG_ECU_201,EQ_PDB_L_1,SLD_MG_ECU_198
498,7,7,7,7,7,7,7,7,6,7,...,TW_CLEARANCE_SNR_ECU_056,SLD_MG_ECU_202,EQ_FUEL_PRESS_SSR_HI-TNGA_E2,SLD_MG_ECU_203,CLEARANCE_SNR_ECU_0890,HV_ECU-HV_0025,EQ_NE_SSR-TNGA_NEP,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,SLD_EFI_ECU-TNGA_KNOCK_SSR_101,SLD_EFI_ECU-TNGA_KNOCK_SSR_102
499,7,7,7,7,7,7,7,7,7,7,...,TW_CLEARANCE_SNR_ECU_056,na,TW_CLEARANCE_SNR_ECU_055,SLD_MG_ECU_200,TW_CLEARANCE_SNR_ECU_062,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,SLD_MG_ECU_201,TW_CLEARANCE_SNR_ECU_056,EQ_PDB_L_5


In [189]:
data_test.to_csv("data/data_test_13.csv", index=False)

In [211]:
def get_bin_10(n,bins10):
    """
    get bin index
    :param n:
    :param bins10:
    :return:
    """
    _bin=10
    for i in range(0,len(bins10)):
        if bins10[i]> n:
            _bin=i-1
            break
    return _bin

def get_pin_signal_test(sig_conn,i,signal_cat):
    """
    #1 number pins [0,Num_Pin_unique_max]
    #2 distance to center pin [0,11]bucket
    #3 number of neighbors[0,Num_Pin_unique_max-1]
    #4 internal/external[0,1]
    #5 mean_neighbors [0,11]bucket
    #6 distance centroide [0,11]bucket
    #7 signal category [0,7]
    #8 color wire categorical[0,max_color_categories]
    #9 thickness wire continuous[0,51]bucket
    #10 multicore yes/no [0,1]
    #11 number of internal pins [0,Num_Pin_unique_max//2]
    #12 nosignal [0,1]
    #13 ismulticore [0,1]
    #14 category multicore [0,max_multicore_categories]
    get observation from File
    :param sig_conn:
    :param i:
    :return:
    """
    # create bins
    bins_distance= np.linspace(sig_conn['1']['min_distance'], sig_conn['1']['max_distance'], 11)
    bins_thickness= np.linspace(sig_conn['1']['Wire_WireCSA_min'], sig_conn['1']['Wire_WireCSA_max'], 51)

    Num_Pins=sig_conn[str(i)]['Num_Pins']
    distance_to_center=get_bin_10(random.randint(0, 10), bins_distance)
    neighbors_length=sig_conn[str(i)]['neighbors_length']
    internal_pin = 1 if sig_conn[str(i)]['internal_pin'] =='True' else 0
    mean_neighbors=get_bin_10(random.randint(0, 10), bins_distance)
    centroide_distance=get_bin_10(sig_conn[str(i)]['centroide_distance'], bins_distance)
    signal_group= signal_cat # Signal Category for test
    Cat_Wire_WireColor=sig_conn[str(i)]['Cat_Wire_WireColor']
    WireCSA=get_bin_10(random.randint(0, 51), bins_thickness)
    is_multicore = 0 if sig_conn[str(i)]['is_multicore'] =='na' else 1
    number_internal_pin= sig_conn[str(i)]['number_internal_pin']
    nosignal = random.randint(0, 1)
    ismulticore= random.randint(0, 1)
    cat_multicore=sig_conn[str(i)]['multicore_same']
    return list([Num_Pins,distance_to_center,neighbors_length,internal_pin,mean_neighbors,
                 centroide_distance,signal_group,Cat_Wire_WireColor,WireCSA,is_multicore,
                 number_internal_pin,nosignal,ismulticore, cat_multicore])

In [216]:
from stable_baselines3.common.vec_env import DummyVecEnv
env_demo = DummyVecEnv([lambda: conn(Num_Pin_unique_max, max_color_categories, max_categories_multicore, Num_Pins)])

In [218]:
data_test['prediction']=0
for i, row in data_test.iterrows():
    env = conn(Num_Pin_unique_max, max_color_categories,max_categories_multicore, Num_Pins) 
    obs = env.reset()
    for key in connectors_dicc[connector].keys():
        if key != 'PartNumber':
    #for c in tcols:
            signalcat= data_test.loc[i, key]
            obs = np.array(get_pin_signal_test(connectors_dicc[connector],key, signalcat))
            action, _state = model.predict(obs, deterministic=False)
        
            obs, reward, done, info = env.step(action.astype(int))
            
            env.render()
            if done:
               obs = env.reset()
    
    print(reward)
    data_test.at[i,'prediction']=reward
#     if i > 2:
#         break    

104
121
136
110
104
116
116
110
116
104
116
115
128
98
128
103
128
91
128
118
116
98
104
134
104
128
98
110
128
128
104
128
128
128
128
128
133
116
128
116
118
104
116
128
98
116
116
116
116
124
128
116
116
124
128
103
128
128
104
116
128
146
116
128
128
128
128
116
115
128
128
128
116
116
133
128
104
92
98
128
104
124
128
116
116
116
116
118
104
104
116
116
128
121
104
104
116
128
128
116
92
128
128
110
116
116
124
128
116
116
88
116
116
116
128
128
128
105
128
128
128
104
110
115
118
116
128
116
116
116
116
128
116
128
128
98
92
128
116
128
128
128
104
128
116
128
128
116
128
146
116
136
116
110
116
110
116
116
128
116
116
104
128
128
128
94
128
128
128
116
128
124
116
128
128
128
128
104
116
128
104
116
116
116
128
116
116
110
121
116
128
109
128
116
116
110
116
116
116
110
116
136
116
104
110
116
128
128
116
128
136
128
128
116
128
124
116
104
116
124
116
128
104
116
116
128
128
146
128
116
136
116
116
116
116
104
128
104
110
128
98
98
128
110
128
128
116
133
98
128
112
104
116
116

In [193]:
obs = env.reset()
for key in connectors_dicc[connector].keys():
    if key != 'PartNumber':
        obs = np.array(get_pin_signal(connectors_dicc[connector],key))
        action, _state = model.predict(obs, deterministic=True)
        
        obs, reward, done, info = env.step(action.astype(int))
        print(reward)
        env.render()
        if done:
           obs = env.reset()
print(reward)

18
18
17
17
22
22
41
41
48
48
55
55
74
74
81
81
82
82
89
89
96
96
103
103
110
110
110


In [214]:
data_test.head()

,1,2,3,4,5,6,7,8,9,10,...,cat_5,cat_6,cat_7,cat_8,cat_9,cat_10,cat_11,cat_12,cat_13,prediction
0,7,7,7,7,7,7,7,7,7,7,...,TW_CLEARANCE_SNR_ECU_062,SLD_EFI_ECU-TNGA_KNOCK_SSR_101,TW_CLEARANCE_SNR_ECU_056,TW_CLEARANCE_SNR_ECU_056,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,na,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,HV_ECU-HV_0025,TW_CLEARANCE_SNR_ECU_055,116
1,7,7,7,7,7,7,7,7,7,7,...,SLD_EFI_ECU-TNGA_KNOCK_SSR_101,EQ_PDB_L_2,EQ_NE_SSR-TNGA_NEP,TW_CLEARANCE_SNR_ECU_062,EQ_NE_SSR-TNGA_NE-,SLD_MG_ECU_200,SLD_MG_ECU_203,TW_CLEARANCE_SNR_ECU_062,TW_CLEARANCE_SNR_ECU_056,128
2,6,7,7,7,7,7,7,7,7,6,...,EQ_NE_SSR-TNGA_VCNE,CLEARANCE_SNR_ECU_0885,EQ_PDB_L_5,SLD_MG_ECU_203,EQ_PDB_L_1,HV_ECU-HV_0024,GND_BK_UP_LP_LH_E,EQ_PDB_L_2,CLEARANCE_SNR_ECU_0885,128
3,7,7,7,7,6,7,7,7,4,7,...,HV_ECU-HV_0025,EQ_NE_SSR-TNGA_VCNE,SLD_MG_ECU_197,TW_CLEARANCE_SNR_ECU_062,GND_BK_UP_LP_LH_E,EQ_PDB_L_1,EQ_PDB_L_2,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,HV_ECU-HV_0024,110
4,7,7,7,7,7,7,7,4,7,7,...,EQ_PDB_L_5,EQ_PDB_L_1,EQ_NE_SSR-TNGA_NE-,GND_BK_UP_LP_LH_E,na,SLD_MG_ECU_201,SLD_MG_ECU_199,CLEARANCE_SNR_ECU_0885,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,128


In [ ]:
print(env.observation_space)
print(env.action_space)
print(env.action_space.sample())

In [ ]:
n_steps = 1300
GO_LEFT=random.randint(0, max_categories )
for step in range(n_steps):
    #print("Step {}".format(step + 1))
    GO_LEFT=random.randint(0, max_categories )
    obs, reward, done, info = env.step(GO_LEFT)
    #print('obs=', obs, 'reward=', reward, 'done=', done)
    env.render()
    if done:
        print("Goal reached!", "reward=", reward)
        obs = env.reset()
        

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
env_train = DummyVecEnv([lambda: conn(connectors_dicc,list_signals ,connector ,len_test_conn, max_categories,state_space)])

In [ ]:
# 13 TO_161_RH_F/169_13P  TO_121_TNGA_F/219_13P

In [ ]:
env_train = DummyVecEnv([lambda: conn(connectors_dicc,list_signals ,connector ,len_test_conn, max_categories,state_space)])

In [ ]:
from stable_baselines3 import DQN, TD3

In [ ]:
from stable_baselines3 import DQN, TD3
model = DQN('MlpPolicy', env_train, verbose=2)

In [ ]:
#model = TD3('MlpPolicy', env_train, verbose=2)

In [ ]:
import time
time.time()

In [ ]:
import time
time1=time.time()
model.learn(10000)
time2=time.time()

print(time2-time1)

In [ ]:
obser=[6,7,8,4,3,5,0,1,2,8,8,8,8,]#'TO_121_TNGA_F/219_13P'
obser=[0,2,1] #TO_111_LH_F/143_3P
#obser=[14,15,27,6,5,7,2,3,4,27,27,27,27] #TO_121_TNGA_F/219_13P   TO_161_RH_F/169_13P 

obser=[13,18,18,21,27,27,12,18,16,22,20,27,27]

In [ ]:
new_observ=list(np.zeros(len(obser), dtype='int'))
reward=0
for x in range(0,len(obser)):
    new_observ[x]=obser[x]
    #print(new_observ)
    pred=model.predict(np.array(new_observ), deterministic= False)
    reward=reward+pred[0]
print(reward)

In [ ]:
model.env.reset()
model.env.observation_space

In [ ]:
model.save("")